## 05b — Cluster-Aware CV Split: Does It Predict Blind Performance Better?

**Question.** Our existing 5x5 CV (`03_features_and_fold_split.ipynb`'s `cv_folds.csv`, RepeatedKFold, seed 42, compound-independent) may be letting `chemprop_chemeleoninit` "cheat" via near-duplicate leakage: if the real blind test set is constructed by hit-expansion (a small set of active hits, each surrounded by close structural analogs -- the leading hypothesis from this project's ongoing leaderboard investigation), a random split can place near-identical analogs on both sides of the train/held-out boundary within the same fold. This notebook builds a **Tanimoto-similarity-clustered** CV split instead -- whole clusters of similar compounds stay together on one side of each fold -- and asks whether it produces a CV estimate that's a more honest predictor of real blind performance than the random split, using `10c`'s real, scored blind result as the yardstick.

**Not the same check as notebook 02's scaffold-split ruling.** `02_chemical_space_exploration.ipynb` Section 5 found Bemis-Murcko scaffolds are 96.4% singletons here -- essentially compound-identity, behaving like random. This notebook uses **Tanimoto fingerprint similarity clustering** (RDKit Butina, threshold 0.4 -- the same threshold Section 4e of notebook 02 already validated as producing a large, stable sample of "similar" pairs, unlike 0.9/0.8) instead of scaffold identity: a finer-grained, threshold-based notion of "similar" that should (and, per Section 2 below, does) produce non-trivial cluster structure even though scaffold clustering didn't.

**What this notebook does:**
1. Build Tanimoto/Butina clusters per isoform, independently, from the frozen ECFP4 fingerprints (`src/features.py`, same function notebook 02 uses -- not recomputed).
2. Merge per-isoform clusters that share a compound into single "coupled groups" (needed because `chemprop_chemeleoninit` is one shared multitask model per fold -- see the flagged decision in Section 3).
3. Randomly assign whole coupled groups to 5 folds, balanced by total compound count, repeated 5x with different seeds -> 25 fold-partitions, structurally analogous to the existing 5x5 design.
4. Refit `chemprop_chemeleoninit` only, full 25-fold run, identical feature representation/architecture/epochs/patience to notebook 05 -- **the only variable under test is the fold assignment.**
5. Compare this cluster-CV's per-isoform ST-RAE against `10c`'s real blind ST-RAE, side by side with notebook 05's original random-CV ST-RAE vs. the same blind numbers -- report only, no significance test between the two CV estimates (they don't share a common fold index, so Ash et al.'s repeated-measures protocol doesn't apply across them).

**What this notebook explicitly does NOT do:** touch `03`'s frozen `cv_folds.csv` or any of `05`'s saved outputs (all reads of those are read-only, confirmed below); recompute or reimplement ECFP4 fingerprinting; run any other model config; touch notebooks 06-09; build or evaluate an ensemble. Nothing downstream of this notebook currently depends on its result -- it is a single, isolated check of whether the split itself is a meaningful contributor to the CV-to-blind gap.

**All outputs are new, under `outputs/05b_cluster_cv_comparison/`** (clustering diagnostics, the new `cluster_cv_folds.csv`, chemprop run artifacts, per-fold scores, and the final comparison table) -- nothing is written back into `outputs/05_cv_comparison/` or `data/folds/`.


In [1]:
import importlib.metadata
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `from src... import ...` works regardless of cwd

import numpy as np
import pandas as pd

from src.features import ecfp4_fingerprints, butina_clusters
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS, MACRO_ENDPOINT_LABEL, REGRESSION_ENDPOINTS

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

PROCESSED = REPO_ROOT / "data" / "processed"
CURATED_PATH = PROCESSED / "train_inhibition_curated.csv"
OUT = REPO_ROOT / "outputs" / "05b_cluster_cv_comparison"
CLUSTER_DIR = OUT / "clustering"
FOLDS_PATH = OUT / "cluster_cv_folds.csv"
SCORE_DIR = OUT / "scores"
ORIGINAL_CV_SUMMARY_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "summary_table.csv"  # read-only

for d in [OUT, CLUSTER_DIR, OUT / "predictions", OUT / "scores", OUT / "chemprop_runs"]:
    d.mkdir(parents=True, exist_ok=True)

ISOFORMS = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]
PIC50_COLS = {iso: f"{iso}_pIC50_direct_inhibition" for iso in ISOFORMS}
METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]
ISOFORM_LABELS = {endpoint: endpoint.split("_")[0] for endpoint in REGRESSION_ENDPOINTS}
ENDPOINT_LABELS = {**ISOFORM_LABELS, MACRO_ENDPOINT_LABEL: MACRO_ENDPOINT_LABEL}

SIMILARITY_THRESHOLD = 0.4  # notebook 02 Section 4e -- validated as producing a large, stable "similar pair" sample
N_REPEATS = 5
N_FOLDS = 5
FOLD_ASSIGNMENT_SEED_BASE = 405  # governs ONLY the random cluster-group -> fold assignment (Section 4) -- distinct
                                  # from CV_SEED_BASE=42 below, which governs training randomness and is reused
                                  # unchanged from notebook 05 so that seed is not a second variable under test
CV_SEED_BASE = 42  # matches scripts/generate_5x5_cv_manifest.py's own CV_SEED_BASE -- reused verbatim (see Section 5)

print(f"python: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas", "rdkit", "chemprop"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"isoforms: {ISOFORMS}")
print(f"similarity threshold: {SIMILARITY_THRESHOLD}, n_repeats: {N_REPEATS}, n_folds: {N_FOLDS}")
print(f"fold-assignment seed base: {FOLD_ASSIGNMENT_SEED_BASE}, training seed base (reused from 05): {CV_SEED_BASE}")


python: 3.11.13
numpy: 1.26.4
pandas: 2.3.3
rdkit: 2026.3.3
chemprop: 2.3.1
REPO_ROOT: /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge
isoforms: ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
similarity threshold: 0.4, n_repeats: 5, n_folds: 5
fold-assignment seed base: 405, training seed base (reused from 05): 42


## 1. Load curated data + frozen ECFP4 fingerprints

Per CLAUDE.md, row counts are logged before any computation. Fingerprints are computed via `ecfp4_fingerprints()` in `src/features.py` -- the exact same shared function notebook 02 calls (ECFP4/Morgan, radius 2, 2048-bit, `includeChirality=False`) -- called fresh here on `canonical_smiles`, which is what "frozen" means in this context: a deterministic function of already-curated, already-validated SMILES, not a cached file (notebook 02 itself never wrote its fingerprints to disk either). No fingerprinting logic is reimplemented.


In [2]:
train = pd.read_csv(CURATED_PATH)
print(f"{CURATED_PATH.name} shape: {train.shape}")
print()
print("per-isoform non-null pIC50 counts (train):")
iso_label_counts = {}
for iso in ISOFORMS:
    n = int(train[PIC50_COLS[iso]].notna().sum())
    iso_label_counts[iso] = n
    print(f"  {iso}: {n}")

n_labels_per_compound = train[[PIC50_COLS[iso] for iso in ISOFORMS]].notna().sum(axis=1)
print()
print("compounds by number of isoforms labelled:")
print(n_labels_per_compound.value_counts().sort_index())
n_multi_isoform = int((n_labels_per_compound >= 2).sum())
print(f"\ncompounds labelled for 2+ isoforms: {n_multi_isoform} ({100 * n_multi_isoform / len(train):.1f}% of {len(train)})")
print("-- this is why per-isoform clusters need merging before fold assignment (Section 3): a compound in this")
print("   group can appear in two different isoforms' independent Butina clustering runs at once.")

fps = ecfp4_fingerprints(train["canonical_smiles"].tolist())
n_fail = sum(fp is None for fp in fps)
print(f"\n{len(fps)} fingerprints generated, {n_fail} parse failures")
assert n_fail == 0, "unexpected fingerprint parse failure on already-curated SMILES -- stopping"


train_inhibition_curated.csv shape: (4905, 20)

per-isoform non-null pIC50 counts (train):
  CYP1A2: 1412
  CYP2C9: 1285
  CYP2D6: 1493
  CYP3A4: 2335

compounds by number of isoforms labelled:
1    3596
2    1039
3     229
4      41
Name: count, dtype: int64

compounds labelled for 2+ isoforms: 1309 (26.7% of 4905)
-- this is why per-isoform clusters need merging before fold assignment (Section 3): a compound in this
   group can appear in two different isoforms' independent Butina clustering runs at once.



4905 fingerprints generated, 0 parse failures


## 2. Butina clustering, per isoform, independently

Each isoform's labelled subset is clustered on its own (`butina_clusters()`, `src/features.py`, added for this notebook) -- clusters never span isoforms, i.e. a CYP1A2 cluster is built only from CYP1A2-labelled compounds, never mixed with CYP2C9's pool. Threshold 0.4 (Tanimoto similarity), matching notebook 02 Section 4e.


In [3]:
iso_isoform_idx = {}  # iso -> np.ndarray of global row-positions with a non-null label for that isoform
iso_clusters_local = {}  # iso -> list of tuples of LOCAL positions (into iso_isoform_idx[iso])
iso_clusters_global = {}  # iso -> list of tuples of GLOBAL row-positions (into `train`)

cluster_summary_rows = []
for iso in ISOFORMS:
    mask = train[PIC50_COLS[iso]].notna().to_numpy()
    idx = np.where(mask)[0]
    sub_fps = [fps[i] for i in idx]
    clusters = butina_clusters(sub_fps, cutoff=SIMILARITY_THRESHOLD)

    iso_isoform_idx[iso] = idx
    iso_clusters_local[iso] = clusters
    iso_clusters_global[iso] = [tuple(idx[i] for i in c) for c in clusters]

    sizes = np.array([len(c) for c in clusters])
    n_singleton = int((sizes == 1).sum())
    cluster_summary_rows.append({
        "isoform": iso,
        "n_compounds": len(idx),
        "n_clusters": len(clusters),
        "singleton_clusters": n_singleton,
        "singleton_cluster_frac_of_clusters": round(n_singleton / len(clusters), 4),
        "compounds_in_singleton_clusters": n_singleton,
        "compounds_in_singleton_frac": round(n_singleton / len(idx), 4),
        "cluster_size_min": int(sizes.min()),
        "cluster_size_median": float(np.median(sizes)),
        "cluster_size_mean": round(float(sizes.mean()), 3),
        "cluster_size_max": int(sizes.max()),
    })
    print(
        f"{iso}: n={len(idx)} compounds -> {len(clusters)} clusters "
        f"(singleton: {n_singleton}/{len(clusters)} clusters = {100*n_singleton/len(clusters):.1f}%, "
        f"{100*n_singleton/len(idx):.1f}% of compounds); "
        f"size min/median/mean/max = {sizes.min()}/{np.median(sizes):.1f}/{sizes.mean():.2f}/{sizes.max()}"
    )
    # every index covered exactly once -- Butina's own contract, verified directly rather than assumed
    covered = sorted(i for c in clusters for i in c)
    assert covered == list(range(len(idx))), f"{iso}: Butina clustering did not partition all {len(idx)} compounds exactly once"

cluster_summary = pd.DataFrame(cluster_summary_rows)
cluster_summary_path = CLUSTER_DIR / "per_isoform_cluster_summary.csv"
cluster_summary.to_csv(cluster_summary_path, index=False)
print(f"\nwrote {cluster_summary_path}")
cluster_summary


CYP1A2: n=1412 compounds -> 1013 clusters (singleton: 724/1013 clusters = 71.5%, 51.3% of compounds); size min/median/mean/max = 1/1.0/1.39/10
CYP2C9: n=1285 compounds -> 757 clusters (singleton: 317/757 clusters = 41.9%, 24.7% of compounds); size min/median/mean/max = 1/2.0/1.70/6


CYP2D6: n=1493 compounds -> 1214 clusters (singleton: 1063/1214 clusters = 87.6%, 71.2% of compounds); size min/median/mean/max = 1/1.0/1.23/19


CYP3A4: n=2335 compounds -> 1105 clusters (singleton: 508/1105 clusters = 46.0%, 21.8% of compounds); size min/median/mean/max = 1/2.0/2.11/19

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/clustering/per_isoform_cluster_summary.csv


,isoform,n_compounds,n_clusters,singleton_clusters,singleton_cluster_frac_of_clusters,compounds_in_singleton_clusters,compounds_in_singleton_frac,cluster_size_min,cluster_size_median,cluster_size_mean,cluster_size_max
0,CYP1A2,1412,1013,724,0.7147,724,0.5127,1,1.0,1.394,10
1,CYP2C9,1285,757,317,0.4188,317,0.2467,1,2.0,1.697,6
2,CYP2D6,1493,1214,1063,0.8756,1063,0.7120,1,1.0,1.230,19
3,CYP3A4,2335,1105,508,0.4597,508,0.2176,1,2.0,2.113,19


**Sanity check against notebook 02, before trusting this structure.** Notebook 02 never built clusters itself (it only characterized whether cluster-based splitting was likely to differ from random), so there's no cluster-count figure there to match exactly -- the check here is qualitative: does this cluster structure look meaningfully *less* singleton-dominated than the scaffold split notebook 02 ruled out (96.4% singleton scaffolds, "behaves like random")? Printed above: CYP2C9 and CYP3A4 are well below that bar (42-46% of clusters are singletons), consistent with 02 Section 4e's finding that both isoforms had thousands of "similar pairs" (>=0.4 Tanimoto) to draw non-trivial cluster structure from. CYP1A2 and especially CYP2D6 run more singleton-heavy (72% and 88% of clusters respectively) -- still meaningfully below scaffold's 96.4%, and every isoform has real, non-trivial clusters (max size 6-19, not just 1-2), but this is reported plainly rather than overstated: the cluster-vs-random-split contrast this design relies on is real but isoform-dependent, strongest for CYP2C9/CYP3A4, weaker for CYP1A2/CYP2D6.


## 3. Merge per-isoform clusters into coupled groups (flagged decision, confirmed with user)

**The gap between the task spec and the model architecture.** Clusters are built independently per isoform (Section 2) -- but `chemprop_chemeleoninit` is **one shared multitask model per fold** (`src/chemprop_screen.py`'s `load_screen_population`/`build_training_csv`: a single train/val/test split, all 4 isoform targets trained jointly in one Chemprop fit). Section 1 showed 1,309 compounds (26.7%) carry labels for 2+ isoforms -- for those compounds, their CYP1A2 cluster and their CYP2C9 cluster were built independently and can disagree about which fold they belong in. A single global split is still required (there is no way to run one shared multitask fit against two different held-out sets at once), so **whenever a compound bridges two isoforms' clusters, those clusters are merged into one "coupled group"** via union-find, and coupled groups (not raw per-isoform clusters) are what gets randomly assigned to folds in Section 4. This is a strict superset of the stated guarantee: if a whole coupled group lands in one fold, then by construction every isoform's own cluster inside it does too, so no isoform's cluster is ever split across train/held-out.

This was flagged to the user before proceeding (not silently picked), with the alternative -- dropping to 4 independent single-task models, one per isoform, each with its own unmerged fold structure -- laid out alongside it. That alternative was rejected: it would change the model itself (multitask -> single-task), conflicting with the task's "the only variable under test is the fold assignment, nothing else." An empirical check (run before asking, so the question came with real numbers rather than a hypothetical) confirmed the merge doesn't blow up into one dominant supercluster on this dataset -- reproduced formally below.


In [4]:
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


uf = UnionFind()
for iso in ISOFORMS:
    for cluster in iso_clusters_global[iso]:
        first = cluster[0]
        uf.find(first)
        for c in cluster[1:]:
            uf.union(first, c)

all_compound_idx = list(train.index)  # every curated compound has >=1 isoform label, confirmed in Section 1

from collections import defaultdict
groups = defaultdict(list)
for i in all_compound_idx:
    groups[uf.find(i)].append(i)
coupled_groups = list(groups.values())  # list of lists of GLOBAL row-positions

group_sizes = sorted((len(g) for g in coupled_groups), reverse=True)
n_merged = sum(1 for s in group_sizes if s > 1)
print(f"labelled compounds covered: {sum(len(g) for g in coupled_groups)} (expected {len(train)})")
print(f"coupled groups: {len(coupled_groups)} total, {n_merged} with >1 compound, {len(coupled_groups) - n_merged} singleton")
print(f"largest 10 coupled-group sizes: {group_sizes[:10]}")
print(f"largest group: {group_sizes[0]} compounds ({100*group_sizes[0]/len(train):.2f}% of all {len(train)} labelled compounds)")
print(f"compounds in groups of size >10: {sum(s for s in group_sizes if s > 10)}")

assert sum(len(g) for g in coupled_groups) == len(train), "coupled groups do not cover every compound exactly once"
assert len(set(i for g in coupled_groups for i in g)) == len(train), "a compound appears in more than one coupled group"


labelled compounds covered: 4905 (expected 4905)
coupled groups: 2764 total, 992 with >1 compound, 1772 singleton
largest 10 coupled-group sizes: [36, 19, 19, 18, 18, 17, 16, 16, 15, 14]
largest group: 36 compounds (0.73% of all 4905 labelled compounds)
compounds in groups of size >10: 573


## 4. Randomized fold assignment: clusters are fixed, fold membership repeats

Coupled groups (from Section 3) are the unit of assignment -- never split across folds. For each of the 5 repeats: groups are shuffled (seeded, so the order differs by repeat) and then walked in that shuffled order, each one going to whichever fold currently holds the fewest total compounds (ties broken by lowest fold index). This is a randomized assignment (the seed controls which groups end up together) that also satisfies "balanced as evenly as possible by total compound count, not cluster count" -- shuffling first means the balance isn't a fixed deterministic outcome, but greedy min-fill still keeps every fold close to `len(train)/5` compounds regardless of shuffle order.

Clustering itself (Sections 2-3) happens once -- only this fold-assignment step is repeated, with 5 different seeds, giving 25 total fold-partitions (`repeat_0`...`repeat_4`, fold values 0-4), structurally analogous to `cv_folds.csv`'s own `repeat_0`...`repeat_4` columns.


In [5]:
def assign_groups_to_folds(groups: list, n_folds: int, seed: int) -> dict:
    """Shuffle `groups` (list of lists of global row-positions) under `seed`, then walk
    them in that order assigning each whole group to the fold with the current smallest
    total compound count. Returns {global_row_position: fold_index}."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(groups))
    fold_totals = [0] * n_folds
    assignment = {}
    for gi in order:
        group = groups[gi]
        f = int(np.argmin(fold_totals))
        for i in group:
            assignment[i] = f
        fold_totals[f] += len(group)
    return assignment, fold_totals


fold_seeds = [int(s.generate_state(1)[0]) for s in np.random.SeedSequence(FOLD_ASSIGNMENT_SEED_BASE).spawn(N_REPEATS)]
print(f"fold-assignment seeds (repeat 0..{N_REPEATS-1}): {fold_seeds}")
assert len(set(fold_seeds)) == N_REPEATS, "fold-assignment seeds are not all distinct"

cluster_cv_folds = train[["Molecule_Name", "inchikey"]].copy()
balance_rows = []
for repeat in range(N_REPEATS):
    assignment, fold_totals = assign_groups_to_folds(coupled_groups, N_FOLDS, fold_seeds[repeat])
    fold_col = np.array([assignment[i] for i in range(len(train))])
    cluster_cv_folds[f"repeat_{repeat}"] = fold_col
    print(f"repeat_{repeat} (seed={fold_seeds[repeat]}): fold compound counts = {fold_totals}")

    for iso in ISOFORMS:
        idx = iso_isoform_idx[iso]
        counts = pd.Series(fold_col[idx]).value_counts().reindex(range(N_FOLDS), fill_value=0)
        for fold, count in counts.items():
            balance_rows.append({"repeat": repeat, "fold": fold, "isoform": iso, "n_labelled_compounds": int(count)})
    for fold in range(N_FOLDS):
        balance_rows.append({"repeat": repeat, "fold": fold, "isoform": "ALL", "n_labelled_compounds": int(fold_totals[fold])})

fold_balance = pd.DataFrame(balance_rows)
fold_balance_path = CLUSTER_DIR / "fold_balance.csv"
fold_balance.to_csv(fold_balance_path, index=False)

assert cluster_cv_folds[[f"repeat_{r}" for r in range(N_REPEATS)]].isin(range(N_FOLDS)).all().all()
cluster_cv_folds.to_csv(FOLDS_PATH, index=False)
print(f"\nwrote {FOLDS_PATH}: {cluster_cv_folds.shape}")
print(f"wrote {fold_balance_path}: {fold_balance.shape}")
cluster_cv_folds.head()


fold-assignment seeds (repeat 0..4): [2696949424, 229610798, 3558783788, 3615032138, 4057354562]
repeat_0 (seed=2696949424): fold compound counts = [979, 980, 990, 978, 978]
repeat_1 (seed=229610798): fold compound counts = [981, 981, 981, 981, 981]
repeat_2 (seed=3558783788): fold compound counts = [982, 981, 981, 981, 980]
repeat_3 (seed=3615032138): fold compound counts = [980, 983, 980, 981, 981]
repeat_4 (seed=4057354562): fold compound counts = [984, 981, 981, 979, 980]

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/cluster_cv_folds.csv: (4905, 7)
wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/clustering/fold_balance.csv: (125, 4)


,Molecule_Name,inchikey,repeat_0,repeat_1,repeat_2,repeat_3,repeat_4
0,OCNT-0000422,RADKZDMFGJYCBB-UHFFFAOYSA-N,0,4,4,1,2
1,OCNT-0001882,TXCGAZHTZHNUAI-UHFFFAOYSA-N,3,4,0,1,0
2,OCNT-0007477,XBJWOGLKABXFJE-UHFFFAOYSA-N,2,4,0,4,4
3,OCNT-0010068,PVNIIMVLHYAWGP-UHFFFAOYSA-N,2,2,3,4,3
4,OCNT-0014841,TYMRLRRVMHJFTF-UHFFFAOYSA-N,0,0,4,2,3


**What this shows.** Global per-fold compound counts (printed above) are balanced as instructed -- greedy min-fill keeps every fold within a handful of compounds of `len(train)/5` ≈ 981. Balance was only targeted at the global level (per the task spec: "balanced... by total compound count per fold, not cluster count"), so per-isoform fold sizes are reported next purely as a sanity check, not a second optimization target.


In [6]:
per_isoform_balance = (
    fold_balance[fold_balance["isoform"] != "ALL"]
    .groupby(["repeat", "isoform"])["n_labelled_compounds"]
    .agg(["min", "max", "mean"])
    .round(1)
)
print("per-isoform labelled-compound count per fold, min/max/mean across the 5 folds within each repeat:")
per_isoform_balance


per-isoform labelled-compound count per fold, min/max/mean across the 5 folds within each repeat:


min  max   mean
repeat isoform                 
0      CYP1A2   255  302  282.4
       CYP2C9   229  279  257.0
       CYP2D6   260  323  298.6
       CYP3A4   442  484  467.0
1      CYP1A2   264  304  282.4
       CYP2C9   233  282  257.0
       CYP2D6   281  315  298.6
       CYP3A4   429  508  467.0
2      CYP1A2   267  295  282.4
       CYP2C9   226  301  257.0
       CYP2D6   272  335  298.6
       CYP3A4   436  498  467.0
3      CYP1A2   264  294  282.4
       CYP2C9   236  288  257.0
       CYP2D6   272  332  298.6
       CYP3A4   440  520  467.0
4      CYP1A2   263  293  282.4
       CYP2C9   237  287  257.0
       CYP2D6   284  320  298.6
       CYP3A4   445  500  467.0

**What this shows.** Global balance (previous cell) is tight (979-990 compounds per fold, vs. an even split of 981). Per-isoform balance is looser, as expected from optimizing only the global total -- e.g. CYP3A4's fold sizes range 429-520 across the 5 repeats (mean 467), an isoform-level spread the global-balance objective doesn't control for. This is reported here rather than corrected: the task spec is explicit that balance is by total compound count, not per-isoform, and the per-isoform spread is a direct, expected consequence of coupled groups (built from cross-isoform label overlap) not lining up evenly against any one isoform's own label distribution.


**Verification, not assumption: does every isoform's own cluster actually stay whole within every fold, in all 25 partitions?** This is the entire point of the split (Section 2's task framing), so it's checked directly against the final `cluster_cv_folds.csv` rather than trusted from the construction logic above.


In [7]:
reloaded_folds = pd.read_csv(FOLDS_PATH)
n_violations = 0
for repeat in range(N_REPEATS):
    fold_col = reloaded_folds[f"repeat_{repeat}"].to_numpy()
    for iso in ISOFORMS:
        for cluster in iso_clusters_global[iso]:
            fold_values = set(fold_col[i] for i in cluster)
            if len(fold_values) > 1:
                n_violations += 1
                print(f"VIOLATION: repeat={repeat} {iso} cluster {cluster} spans folds {fold_values}")

print(f"checked {N_REPEATS} repeats x {len(ISOFORMS)} isoforms x (their own cluster set) for train/held-out splitting")
print(f"violations found: {n_violations}")
assert n_violations == 0, "a cluster was split across folds -- the core guarantee of this split failed, stopping"
print("PASS: every isoform's own Butina cluster is wholly contained in one fold, in all 25 fold-partitions.")


checked 5 repeats x 4 isoforms x (their own cluster set) for train/held-out splitting
violations found: 0
PASS: every isoform's own Butina cluster is wholly contained in one fold, in all 25 fold-partitions.


## 5. Refit `chemprop_chemeleoninit`, full 25-fold run (not run inside this notebook)

Trained via `scripts/05b_run_cluster_cv.py`, not inside this notebook or an agentic session -- matching this project's established convention (`04a_baseline_screen.ipynb` Section 4, `train_log2fc_encoder.py`) of running multi-hour Chemprop jobs as a standalone background script, tailed via its own log file, rather than blocking a notebook cell:

```
cd /path/to/OpenADMET-CYP-Blind-Challenge
caffeinate -i nohup /Users/codiefreeman/miniconda3-arm64/envs/cyp-admet-v2/bin/python \
    scripts/05b_run_cluster_cv.py > logs/05b_cluster_cv_stdout.log 2>&1 &
```
Progress: `tail -f logs/05b_cluster_cv.log`.

**Everything except the fold assignment is copied verbatim from notebook 05's own run**: `--from-foundation CHEMELEON --multi-hot-atom-featurizer-mode V2` architecture, 50 epochs / 5 patience, `val_fraction=0.15`, and -- specifically so training randomness isn't a second variable under test alongside the fold assignment -- the exact same 25 per-(repeat,fold) seeds as notebook 05 (`np.random.SeedSequence(42).spawn(25)`, same enumeration order), regenerated by formula and cross-checked at startup against `outputs/05_cv_comparison/manifest.csv` (read-only) to confirm they match exactly. Only `FOLDS_PATH` (this notebook's `cluster_cv_folds.csv` instead of `data/folds/cv_folds.csv`) and the output directory (`outputs/05b_cluster_cv_comparison/` instead of `outputs/05_cv_comparison/`) differ. Same resumability contract as `run_5x5_cv_comparison.py`: a fold's prediction+score CSVs only appear on disk after a full, successful fit, so an interrupted run loses no completed work and retries cleanly.

This section only loads the script's completed output below -- if `outputs/05b_cluster_cv_comparison/scores/` doesn't have 25 files, run the command above first.


## 6. Aggregate the 25 raw score files

Same aggregation notebook 05 Section 1 uses on its own scores, applied here to this run's own (separate) score files: each raw score CSV has 5,000 rows (5 endpoints x 1,000 bootstrap samples) for one (repeat, fold); the point estimate per (repeat, fold, endpoint, metric) is the **mean** of that 1,000-sample bootstrap distribution (matching notebook 05's own precedent, not the median). Collapses to one row per (repeat, fold) -- 25 rows.


In [8]:
score_files = sorted(SCORE_DIR.glob("*.csv"))
print(f"found {len(score_files)} score files in {SCORE_DIR}")
if len(score_files) < N_REPEATS * N_FOLDS:
    raise FileNotFoundError(
        f"expected {N_REPEATS * N_FOLDS} score files, found {len(score_files)} -- "
        "run scripts/05b_run_cluster_cv.py (Section 5) to completion first."
    )

shapes_seen = set()
rows = []
for f in score_files:
    df = pd.read_csv(f)
    shapes_seen.add(df.shape)

    config = df["config"].iloc[0]
    repeat = int(df["repeat"].iloc[0])
    fold = int(df["fold"].iloc[0])
    bootstrap_seed = int(df["bootstrap_seed"].iloc[0])

    point_estimates = df.groupby("Endpoint")[METRIC_NAMES].mean()  # mean, not median -- matches notebook 05 Section 1
    row = {"config": config, "repeat": repeat, "fold": fold, "bootstrap_seed": bootstrap_seed}
    for endpoint, label in ENDPOINT_LABELS.items():
        for metric in METRIC_NAMES:
            row[f"{label}_{metric}"] = point_estimates.loc[endpoint, metric]
    rows.append(row)

print(f"distinct (rows, cols) shapes across all {len(score_files)} files: {shapes_seen}")

cluster_cv_summary = pd.DataFrame(rows).sort_values(["repeat", "fold"]).reset_index(drop=True)
print(f"rows before/after aggregation: {len(score_files)} files -> {len(cluster_cv_summary)} summary rows")

assert len(cluster_cv_summary) == N_REPEATS * N_FOLDS, f"expected {N_REPEATS * N_FOLDS} rows, got {len(cluster_cv_summary)}"
assert cluster_cv_summary[["repeat", "fold"]].drop_duplicates().shape[0] == N_REPEATS * N_FOLDS, "duplicate (repeat, fold) rows found"
assert cluster_cv_summary["config"].eq("chemprop_chemeleoninit").all(), "unexpected config value in score files"
assert not cluster_cv_summary.isna().any().any(), "unexpected NaN in aggregated summary table"

cluster_cv_summary_path = OUT / "cluster_cv_summary_table.csv"
cluster_cv_summary.to_csv(cluster_cv_summary_path, index=False)
print(f"wrote {cluster_cv_summary_path}: {cluster_cv_summary.shape}")
cluster_cv_summary.head()


found 25 score files in /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/scores
distinct (rows, cols) shapes across all 25 files: {(5000, 11)}
rows before/after aggregation: 25 files -> 25 summary rows
wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/cluster_cv_summary_table.csv: (25, 29)


,config,repeat,fold,bootstrap_seed,CYP1A2_ST-RAE,CYP1A2_MAE,CYP1A2_R2,CYP1A2_Spearman_R,CYP1A2_Kendall_Tau,CYP2C9_ST-RAE,CYP2C9_MAE,CYP2C9_R2,CYP2C9_Spearman_R,CYP2C9_Kendall_Tau,CYP2D6_ST-RAE,CYP2D6_MAE,CYP2D6_R2,CYP2D6_Spearman_R,CYP2D6_Kendall_Tau,CYP3A4_ST-RAE,CYP3A4_MAE,CYP3A4_R2,CYP3A4_Spearman_R,CYP3A4_Kendall_Tau,MA_ST-RAE,MA_MAE,MA_R2,MA_Spearman_R,MA_Kendall_Tau
0,chemprop_chemeleoninit,0,0,2684470948,0.899689,0.644262,0.224269,0.448799,0.314427,0.707684,0.482886,0.370586,0.583460,0.419778,1.104741,0.646083,0.019090,0.274056,0.187014,0.512614,0.564088,0.564632,0.772191,0.576720,0.806182,0.584330,0.294644,0.519626,0.374485
1,chemprop_chemeleoninit,0,1,4091952314,0.943093,0.738308,0.185509,0.564353,0.390106,0.688509,0.444634,0.393973,0.627289,0.455089,0.990487,0.580553,0.116810,0.422440,0.292141,0.510456,0.511085,0.598981,0.767373,0.576144,0.783136,0.568645,0.323818,0.595364,0.428370
2,chemprop_chemeleoninit,0,2,233227757,0.754951,0.605228,0.326909,0.525215,0.368350,0.718628,0.483833,0.386859,0.634165,0.456179,1.197627,0.665402,-0.054814,0.241905,0.160457,0.546987,0.529443,0.557173,0.757995,0.569429,0.804548,0.570977,0.304032,0.539820,0.388604
3,chemprop_chemeleoninit,0,3,3276785861,0.910358,0.645148,0.209099,0.481242,0.339404,0.603479,0.465757,0.389613,0.650143,0.468599,0.910407,0.591632,0.111046,0.388639,0.267170,0.715806,0.667589,0.438197,0.692961,0.503010,0.785012,0.592531,0.286989,0.553246,0.394546
4,chemprop_chemeleoninit,0,4,3644269654,0.788169,0.691310,0.263558,0.511334,0.360359,0.631167,0.468784,0.403228,0.664420,0.478345,0.887421,0.679157,0.176058,0.476335,0.329210,0.477775,0.558736,0.595886,0.779484,0.586664,0.696133,0.599497,0.359683,0.607893,0.438644


## 7. Does the cluster-CV estimate sit closer to the real blind result?

Per isoform, two gaps against `10c`'s real, scored blind ST-RAE (the same plain, unensembled `chemprop_chemeleoninit` config, full retrain, `cyp-admet-v2` -- see CLAUDE.md's leaderboard-submissions log):

- **gap_original** = `|05's original random-CV ST-RAE − 10c's blind ST-RAE|`
- **gap_cluster** = `|this cluster-CV ST-RAE − 10c's blind ST-RAE|`

`05's original random-CV ST-RAE` is read directly from `outputs/05_cv_comparison/summary_table.csv` (read-only, never modified by this notebook), filtered to `config == "chemprop_chemeleoninit"`, mean over its own 25 folds -- not hardcoded, so this stays correct if that file is ever regenerated. `10c`'s blind numbers are hardcoded from the task brief / CLAUDE.md's leaderboard log, since there is no live file to read them from.

**Report only -- no significance test between (a) and (b).** They don't share a common fold index (different fold structures entirely), so Ash et al.'s repeated-measures protocol, used throughout notebook 05, doesn't apply across them -- only within each split's own 25 folds. This section reports the two point estimates and their gaps side by side; it does not claim one is statistically distinguishable from the other.


In [9]:
# 10c's real, scored blind ST-RAE (per-isoform) -- hardcoded from the task brief / CLAUDE.md's
# leaderboard-submissions log (docs/leaderboard_submissions.md); no live file holds these.
BLIND_ST_RAE_10C = {
    "CYP1A2": 0.6954,
    "CYP2C9": 0.5489,
    "CYP2D6": 1.1673,
    "CYP3A4": 0.4434,
}

original_cv = pd.read_csv(ORIGINAL_CV_SUMMARY_PATH)  # read-only -- outputs/05_cv_comparison/summary_table.csv
print(f"loaded (read-only) {ORIGINAL_CV_SUMMARY_PATH}: {original_cv.shape}")
original_cv_chemeleon = original_cv[original_cv["config"] == "chemprop_chemeleoninit"]
assert len(original_cv_chemeleon) == 25, f"expected 25 rows for chemprop_chemeleoninit in 05's summary, got {len(original_cv_chemeleon)}"

comparison_rows = []
for iso in ISOFORMS:
    col = f"{iso}_ST-RAE"
    original_st_rae = float(original_cv_chemeleon[col].mean())
    cluster_st_rae = float(cluster_cv_summary[col].mean())
    blind_st_rae = BLIND_ST_RAE_10C[iso]
    gap_original = abs(original_st_rae - blind_st_rae)
    gap_cluster = abs(cluster_st_rae - blind_st_rae)
    comparison_rows.append({
        "isoform": iso,
        "original_random_cv_ST-RAE": round(original_st_rae, 4),
        "cluster_cv_ST-RAE": round(cluster_st_rae, 4),
        "blind_10c_ST-RAE": blind_st_rae,
        "gap_original": round(gap_original, 4),
        "gap_cluster": round(gap_cluster, 4),
        "smaller_gap": "cluster" if gap_cluster < gap_original else ("original" if gap_original < gap_cluster else "tie"),
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_path = OUT / "comparison_vs_10c.csv"
comparison_table.to_csv(comparison_path, index=False)
print(f"\nwrote {comparison_path}")
comparison_table


loaded (read-only) /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05_cv_comparison/summary_table.csv: (300, 29)

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/comparison_vs_10c.csv


,isoform,original_random_cv_ST-RAE,cluster_cv_ST-RAE,blind_10c_ST-RAE,gap_original,gap_cluster,smaller_gap
0,CYP1A2,0.8685,0.8670,0.6954,0.1731,0.1716,cluster
1,CYP2C9,0.6787,0.6666,0.5489,0.1298,0.1177,cluster
2,CYP2D6,1.0039,1.0013,1.1673,0.1634,0.1660,original
3,CYP3A4,0.5461,0.5586,0.4434,0.1027,0.1152,original


**What this shows: a mixed, small-magnitude result -- not a clean win for the cluster-aware split.**

| isoform | original random-CV ST-RAE | cluster-CV ST-RAE | 10c blind ST-RAE | gap_original | gap_cluster | smaller gap |
|---|---|---|---|---|---|---|
| CYP1A2 | 0.8685 | 0.8670 | 0.6954 | 0.1731 | 0.1716 | cluster |
| CYP2C9 | 0.6787 | 0.6666 | 0.5489 | 0.1298 | 0.1177 | cluster |
| CYP2D6 | 1.0039 | 1.0013 | 1.1673 | 0.1634 | 0.1660 | original |
| CYP3A4 | 0.5461 | 0.5586 | 0.4434 | 0.1027 | 0.1152 | original |

CYP1A2 and CYP2C9 move (slightly) closer to the real blind result under the cluster-aware split; CYP2D6 and CYP3A4 move (slightly) further away. In every case the *change* in gap (0.001-0.03 ST-RAE) is small relative to the gap itself (0.10-0.17) -- the cluster split does not close most of the CV-to-blind gap for any isoform, on either side of the split. Both random-CV and cluster-CV badly underestimate CYP2D6's blind ST-RAE in absolute terms (both ~1.00 vs. blind 1.17) -- the cluster split doesn't fix this either, consistent with the assay-construction explanation from 04c/10b (Section 8 below), which a fold-assignment change wouldn't be expected to touch.

**Read plainly, this is a 2-2 split with small effect sizes, not evidence that cluster-aware CV is (or isn't) a more honest predictor of blind performance here.** Per this notebook's scope, no significance test was run between the two CV estimates (they don't share a fold structure), so "2 of 4 isoforms move closer, by a small amount" is the full, honest summary -- not rounded up to a directional conclusion in either direction. This is one model, one dataset, one random cluster-to-fold draw per repeat; it doesn't rule out the hit-expansion-leakage hypothesis this notebook set out to test, but it doesn't confirm it either.


## 8. CYP2D6 — flagged, not resolved

CYP2D6 is the one isoform that already runs backward on CV-vs-blind under the random split: notebook 04c/10b's investigation found it independently confirmed (via OpenADMET Discord, Sean Colby) to use a different assay (Echo-MS vs. fluorescence) and to share its blind test compounds with the other three isoforms rather than being separately hit-expansion-seeded. There is no strong prior for what a hit-expansion-motivated split should do to an isoform that was never itself hit-expansion-seeded -- it may move with the other three isoforms, or it may not. This section reports which happened, plainly, without attempting to explain it here (that would be new investigative work outside this notebook's scope).


In [10]:
other_isoforms = [iso for iso in ISOFORMS if iso != "CYP2D6"]
other_direction = comparison_table.set_index("isoform").loc[other_isoforms, "smaller_gap"]
cyp2d6_direction = comparison_table.set_index("isoform").loc["CYP2D6", "smaller_gap"]

print("smaller-gap direction (which CV estimate sits closer to 10c's blind result), per isoform:")
print(comparison_table.set_index("isoform")["smaller_gap"])
print()
same_as_others = other_direction.nunique() == 1 and cyp2d6_direction == other_direction.iloc[0]
print(f"CYP1A2/CYP2C9/CYP3A4 direction(s): {sorted(other_direction.unique())}")
print(f"CYP2D6 direction: {cyp2d6_direction!r}")
print(f"CYP2D6 moves in the SAME direction as the other three: {same_as_others}")
print()
print("Reported plainly, per Section 8's framing above -- no interpretation of WHY attempted here.")


smaller-gap direction (which CV estimate sits closer to 10c's blind result), per isoform:
isoform
CYP1A2     cluster
CYP2C9     cluster
CYP2D6    original
CYP3A4    original
Name: smaller_gap, dtype: object

CYP1A2/CYP2C9/CYP3A4 direction(s): ['cluster', 'original']
CYP2D6 direction: 'original'
CYP2D6 moves in the SAME direction as the other three: False

Reported plainly, per Section 8's framing above -- no interpretation of WHY attempted here.


**What this shows.** CYP2D6 does *not* move with the other three isoforms as a block -- because the other three don't move as a block either: CYP1A2 and CYP2C9 sit closer to blind under the cluster split, while CYP2D6 **and CYP3A4** both sit closer to blind under the original random split. CYP2D6 pairs with CYP3A4 here, not standing alone -- so this result doesn't cleanly support "CYP2D6 is uniquely different because it wasn't hit-expansion-seeded" as an explanation for this particular pattern (CYP3A4 presumably *was* hit-expansion-seeded, per the other three isoforms, yet lands on the same side as CYP2D6). Flagged as stated at the top of this section -- no attempt is made here to explain why, only to report the actual grouping plainly rather than the cleaner CYP2D6-vs-the-rest story the setup hypothesized.


## Summary

- Built a Tanimoto/Butina-clustered (threshold 0.4) 5x5 CV split for `chemprop_chemeleoninit`, entirely new files/outputs, without touching notebook 03's `cv_folds.csv` or any of notebook 05's saved results.
- Clusters are meaningfully non-trivial for CYP2C9/CYP3A4 (42-46% singleton), less so for CYP1A2/CYP2D6 (72-88% singleton) -- still well below the 96.4%-singleton scaffold split notebook 02 already ruled out.
- Per-isoform clusters had to be merged into cross-isoform "coupled groups" (union-find) before fold assignment, since 26.7% of compounds carry 2+ isoform labels and `chemprop_chemeleoninit` trains one shared multitask model per fold -- flagged and confirmed with the user before proceeding; verified empirically to not produce a runaway supercluster (largest group 0.73% of the data), and formally verified after construction that no isoform's own cluster is ever split across train/held-out in any of the 25 fold-partitions.
- Refit full 25-fold `chemprop_chemeleoninit`, identical architecture/epochs/patience/feature representation and identical per-(repeat,fold) training seeds to notebook 05 -- the fold assignment is the only variable that changed.
- **Result: mixed, small-magnitude.** CYP1A2/CYP2C9 CV-to-blind gaps shrink slightly under the cluster split; CYP2D6/CYP3A4 gaps widen slightly. No isoform's gap closes substantially either way. CYP2D6 pairs with CYP3A4 in this pattern rather than standing alone, which complicates (without resolving) the "CYP2D6 is different because it wasn't hit-expansion-seeded" framing this notebook set out from.
- **Not concluded**: that the split itself is (or isn't) a material contributor to this project's CV-to-blind gap. The result here is inconclusive at the effect sizes observed, from a single model and a single random draw per repeat -- consistent with the brief's framing that nothing downstream currently depends on this notebook's result. Per CLAUDE.md's own threshold-discipline rule (carried over from notebook 11b/12b's precedent), a real adoption decision would need a pre-stated effect-size threshold this small a difference doesn't clear, not just a directional reading of which side of a 2-2 split is "better."
